In [ ]:
# 1. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# 2. Install required libraries
!pip install pyrogram tgcrypto yt-dlp nest-asyncio -q

import yt_dlp
import os
import nest_asyncio
import shutil
from pyrogram import Client, idle

# Patch asyncio for Colab
nest_asyncio.apply()

from google.colab import userdata
API_ID = int(userdata.get('API_ID'))
API_HASH = userdata.get('API_HASH')
MY_USER_ID = int(userdata.get('USER_ID'))

DRIVE_FOLDER = '/content/drive/MyDrive/BotData'
LOCAL_FOLDER = '/content'
COOKIE_PATH = f'{DRIVE_FOLDER}/cookies.txt'

SESSION_NAME = "userbot_session"
DRIVE_SESSION = f"{DRIVE_FOLDER}/{SESSION_NAME}.session"
LOCAL_SESSION = f"{LOCAL_FOLDER}/{SESSION_NAME}.session"

os.makedirs(DRIVE_FOLDER, exist_ok=True)

# Copy session from Drive to Local if it exists to avoid SQLite lock errors
if os.path.exists(DRIVE_SESSION):
    shutil.copy(DRIVE_SESSION, LOCAL_SESSION)

# Pyrogram now runs out of the fast local storage
app = Client(SESSION_NAME, api_id=API_ID, api_hash=API_HASH, workdir=LOCAL_FOLDER)

user_states = {}

@app.on_message()
async def handle_message(client, message):
    # Only responds to Daredevil (Your Main Account ID)
    if not message.from_user or message.from_user.id != MY_USER_ID:
        return

    chat_id = message.chat.id
    text = message.text

    if text and text.startswith("http"):
        user_states[chat_id] = text
        await message.reply_text("Link received!\n\nType the quality you want: `720`, `480`, `360`, or `270`")
        return

    if chat_id in user_states and text in ["720", "480", "360", "270"]:
        url = user_states[chat_id]
        del user_states[chat_id]

        status_msg = await message.reply_text("Downloading to Google Colab...")
        format_str = f'best[height<={text}]/best'

        ydl_opts = {
            'outtmpl': f'/content/media_{chat_id}.%(ext)s',
            'format': format_str,
            'quiet': True,
            'cookiefile': COOKIE_PATH,
        }

        try:
            with yt_dlp.YoutubeDL(ydl_opts) as ydl:
                info_dict = ydl.extract_info(url, download=True)
                filename = ydl.prepare_filename(info_dict)

            await status_msg.edit_text("Uploading video directly to this chat (Up to 2GB)...")

            # Sends the video right back to the chat you sent the link from!
            await client.send_video(
                chat_id=chat_id,
                video=filename,
                caption=f"Source: {url}"
            )

            os.remove(filename)
            await status_msg.edit_text("✅ Transfer Complete!")

        except Exception as e:
            await status_msg.edit_text(f"Failed: {str(e)}")

async def main():
    print("Starting Pyrogram...")
    await app.start()

    # Automatically backup the session file to Google Drive upon first login
    if not os.path.exists(DRIVE_SESSION) and os.path.exists(LOCAL_SESSION):
        shutil.copy(LOCAL_SESSION, DRIVE_SESSION)
        print("Session successfully backed up to Google Drive!")

    print("Bot is ready! Send a link from Daredevil to Account B, then type the quality.")
    await idle()

await main()